In [ ]:
%run ./phase2_common

In [ ]:
from datetime import date

from pyspark.sql.window import Window

dbutils.widgets.text("batch_id", "", "Batch ID (blank = latest Silver batch)")
dbutils.widgets.dropdown("run_optimize", "true", ["true", "false"], "Run OPTIMIZE after load?")

batch_id = dbutils.widgets.get("batch_id").strip()
run_optimize = dbutils.widgets.get("run_optimize") == "true"
SILVER_MOVIES = f"{SCHEMA}.silver_movies_cleansed"

if not batch_id:
    latest = (
        spark.table(SILVER_MOVIES).groupBy("source_batch_id")
        .agg(F.max("source_extracted_at").alias("ts")).orderBy(F.desc("ts")).first()
    )
    if latest is None:
        dbutils.notebook.exit("NO_SILVER_DATA")
    batch_id = latest["source_batch_id"]

batch_movies = spark.table(SILVER_MOVIES).filter(F.col("source_batch_id") == batch_id)
if not batch_movies.limit(1).count():
    dbutils.notebook.exit("NO_BATCH_MOVIES")
batch_movies.createOrReplaceTempView("gold_batch_movies")

bronze_row = spark.table(BRONZE).filter(F.col("batch_id") == batch_id).select("load_type").first()
load_type = bronze_row[0] if bronze_row else "unknown"
print("Gold build for batch:", batch_id, "| load type:", load_type)


def create_table(name, columns, cluster_by=None):
    ddl = f"CREATE TABLE IF NOT EXISTS {SCHEMA}.{name} ({columns}) USING DELTA"
    try:
        spark.sql(f"{ddl} CLUSTER BY ({cluster_by})" if cluster_by else ddl)
    except Exception as exc:
        print(f"{name}: liquid clustering unavailable ({str(exc)[:80]}); using a plain Delta table")
        spark.sql(ddl)


def step(table, build):
    counts = run_logged(
        "Silver-to-Gold", load_type, f"batch_id={batch_id}; table={table}", batch_id, build
    )
    print(f"{table:<30} {counts}")

In [ ]:
def build_dim_date():
    create_table("gold_dim_date", """
        date_key INT, full_date DATE, day_of_month INT, month_number INT,
        month_name STRING, quarter_number INT, year_number INT, day_of_week INT,
        day_name STRING, is_weekend BOOLEAN
    """, "date_key")
    table = f"{SCHEMA}.gold_dim_date"
    if spark.table(table).count() == (date(2100, 12, 31) - date(1900, 1, 1)).days + 2:
        return {"processed": 0}

    calendar = spark.sql(
        "SELECT explode(sequence(DATE'1900-01-01', DATE'2100-12-31', INTERVAL 1 DAY)) AS d"
    ).select(
        F.date_format("d", "yyyyMMdd").cast("int").alias("date_key"),
        F.col("d").alias("full_date"),
        F.dayofmonth("d").alias("day_of_month"),
        F.month("d").alias("month_number"),
        F.date_format("d", "MMMM").alias("month_name"),
        F.quarter("d").alias("quarter_number"),
        F.year("d").alias("year_number"),
        F.dayofweek("d").alias("day_of_week"),
        F.date_format("d", "EEEE").alias("day_name"),
        F.dayofweek("d").isin(1, 7).alias("is_weekend"),
    )
    unknown = spark.sql("""
        SELECT -1 AS date_key, CAST(NULL AS DATE) AS full_date, CAST(NULL AS INT) AS day_of_month,
               CAST(NULL AS INT) AS month_number, 'Unknown' AS month_name,
               CAST(NULL AS INT) AS quarter_number, CAST(NULL AS INT) AS year_number,
               CAST(NULL AS INT) AS day_of_week, 'Unknown' AS day_name, false AS is_weekend
    """)
    return upsert(table, calendar.unionByName(unknown), ["date_key"], insert_only=True)


def build_dim_movie():
    create_table("gold_dim_movie", """
        movie_key BIGINT, movie_id BIGINT, title STRING, original_title STRING,
        original_language STRING, release_date DATE, release_date_key INT,
        runtime INT, status STRING, adult BOOLEAN, budget_tier STRING,
        version INT, valid_from TIMESTAMP, valid_to TIMESTAMP,
        is_current BOOLEAN, attribute_hash STRING
    """, "movie_key")
    table = f"{SCHEMA}.gold_dim_movie"
    tracked = [
        "title", "original_title", "original_language", "release_date",
        "runtime", "status", "adult", "budget_tier",
    ]
    versions = spark.table(table).groupBy("movie_id").agg(F.max("version").alias("max_version"))
    source = (
        spark.table("gold_batch_movies")
        .select(
            "movie_id", *tracked,
            F.coalesce(F.date_format("release_date", "yyyyMMdd").cast("int"), F.lit(-1)).alias("release_date_key"),
            F.sha2(F.concat_ws("|", *[F.coalesce(F.col(c).cast("string"), F.lit("~")) for c in tracked]), 256)
            .alias("attribute_hash"),
            F.col("source_extracted_at").alias("valid_from"),
        )
        .join(versions, "movie_id", "left")
        .withColumn("version", F.coalesce(F.col("max_version"), F.lit(0)) + 1)
        .drop("max_version")
    )
    source.createOrReplaceTempView("gold_movie_source")

    before = table_version(table)
    spark.sql(f"""
        MERGE INTO {table} t USING gold_movie_source s
        ON t.movie_id = s.movie_id AND t.is_current = true
        WHEN MATCHED AND t.attribute_hash <> s.attribute_hash AND s.valid_from > t.valid_from THEN
            UPDATE SET t.is_current = false, t.valid_to = s.valid_from
    """)
    closed = merge_metrics(table, before)["updated"]
    before = table_version(table)
    spark.sql(f"""
        MERGE INTO {table} t USING gold_movie_source s
        ON t.movie_id = s.movie_id AND t.is_current = true
        WHEN NOT MATCHED THEN INSERT (
            movie_key, movie_id, title, original_title, original_language, release_date,
            release_date_key, runtime, status, adult, budget_tier, version, valid_from,
            valid_to, is_current, attribute_hash
        ) VALUES (
            xxhash64(concat(s.movie_id, '#', s.version)), s.movie_id, s.title, s.original_title,
            s.original_language, s.release_date, s.release_date_key, s.runtime, s.status,
            s.adult, s.budget_tier, s.version, s.valid_from, NULL, true, s.attribute_hash
        )
    """)
    return {"processed": source.count(), "inserted": merge_metrics(table, before)["inserted"], "updated": closed}


step("gold_dim_date", build_dim_date)
step("gold_dim_movie", build_dim_movie)

In [ ]:
def build_dim(table, source, id_col, attrs, key_col):
    columns = f"{key_col} BIGINT, {id_col} BIGINT, " + ", ".join(f"{name} STRING" for name in attrs)
    create_table(table, columns)
    newest = Window.partitionBy(id_col).orderBy(F.desc("source_extracted_at"), F.desc("load_timestamp"))
    rows = (
        spark.table(f"{SCHEMA}.{source}").filter(F.col(id_col).isNotNull())
        .withColumn("rn", F.row_number().over(newest)).filter("rn = 1")
        .select(
            F.xxhash64(id_col).alias(key_col), id_col,
            *[F.col(src).alias(name) for name, src in attrs.items()],
        )
    )
    return upsert(f"{SCHEMA}.{table}", rows, [key_col])


DIMENSIONS = [
    ("gold_dim_genre", "silver_movie_genres", "genre_id", {"genre_name": "genre_name"}, "genre_key"),
    ("gold_dim_studio", "silver_movie_studios", "company_id",
     {"company_name": "company_name", "origin_country": "origin_country"}, "studio_key"),
    ("gold_dim_person", "silver_movie_cast", "person_id", {"person_name": "cast_name"}, "person_key"),
    ("gold_dim_keyword", "silver_movie_keywords", "keyword_id", {"keyword_name": "keyword_name"}, "keyword_key"),
]
for dim in DIMENSIONS:
    step(dim[0], lambda dim=dim: build_dim(*dim))

In [ ]:
current_movies = (
    spark.table(f"{SCHEMA}.gold_dim_movie").filter("is_current = true")
    .join(spark.table("gold_batch_movies").select("movie_id"), "movie_id")
    .select("movie_key", "movie_id")
)
batch_keys = sql_list(r[0] for r in current_movies.select("movie_key").collect())


def build_bridge(table, child, id_col, key_col, merge_keys, columns, extras=()):
    create_table(table, columns, "movie_key")
    rows = (
        current_movies.join(spark.table(f"{SCHEMA}.{child}"), "movie_id")
        .filter(F.col(id_col).isNotNull())
        .select("movie_key", F.xxhash64(id_col).alias(key_col), *extras)
        .distinct()
    )
    return upsert(f"{SCHEMA}.{table}", rows, merge_keys, delete_where=f"t.movie_key IN ({batch_keys})")


BRIDGES = [
    ("gold_bridge_movie_genre", "silver_movie_genres", "genre_id", "genre_key",
     ["movie_key", "genre_key"], "movie_key BIGINT, genre_key BIGINT", ()),
    ("gold_bridge_movie_studio", "silver_movie_studios", "company_id", "studio_key",
     ["movie_key", "studio_key"], "movie_key BIGINT, studio_key BIGINT", ()),
    ("gold_bridge_movie_keyword", "silver_movie_keywords", "keyword_id", "keyword_key",
     ["movie_key", "keyword_key"], "movie_key BIGINT, keyword_key BIGINT", ()),
    ("gold_bridge_movie_cast", "silver_movie_cast", "person_id", "person_key",
     ["movie_key", "credit_id"],
     "movie_key BIGINT, person_key BIGINT, credit_id STRING, character STRING, cast_order INT",
     ("credit_id", "character", "cast_order")),
]
for bridge in BRIDGES:
    step(bridge[0], lambda bridge=bridge: build_bridge(*bridge))

In [ ]:
def build_fact_movie_snapshot():
    create_table("gold_fact_movie_snapshot", """
        snapshot_key BIGINT, movie_key BIGINT, movie_id BIGINT, snapshot_date_key INT,
        batch_id STRING, source_extracted_at TIMESTAMP, popularity DOUBLE,
        vote_average DOUBLE, vote_count BIGINT, budget BIGINT, revenue BIGINT,
        profit BIGINT, roi DOUBLE, review_count BIGINT, avg_review_rating DOUBLE,
        cast_count BIGINT, genre_count BIGINT, studio_count BIGINT, keyword_count BIGINT,
        load_timestamp TIMESTAMP
    """, "snapshot_date_key, movie_key")

    df = (
        spark.table("gold_batch_movies")
        .join(current_movies, "movie_id")
        .join(
            spark.table(f"{SCHEMA}.silver_movie_reviews").groupBy("movie_id").agg(
                F.count("*").alias("review_count"), F.avg("rating").alias("avg_review_rating")
            ),
            "movie_id", "left",
        )
    )
    counted = [
        ("cast_count", "silver_movie_cast", "credit_id"),
        ("genre_count", "silver_movie_genres", "genre_id"),
        ("studio_count", "silver_movie_studios", "company_id"),
        ("keyword_count", "silver_movie_keywords", "keyword_id"),
    ]
    for name, table, column in counted:
        df = df.join(
            spark.table(f"{SCHEMA}.{table}").groupBy("movie_id").agg(F.countDistinct(column).alias(name)),
            "movie_id", "left",
        )
    rows = (
        df.fillna(0, ["review_count"] + [name for name, _, _ in counted])
        .withColumn("snapshot_date_key", F.date_format("source_extracted_at", "yyyyMMdd").cast("int"))
        .withColumn("snapshot_key", F.xxhash64("movie_id", "snapshot_date_key"))
        .withColumn("batch_id", F.col("source_batch_id"))
        .withColumn("load_timestamp", F.current_timestamp())
        .select(
            "snapshot_key", "movie_key", "movie_id", "snapshot_date_key", "batch_id",
            "source_extracted_at", "popularity", "vote_average", "vote_count", "budget",
            "revenue", "profit", "roi", "review_count", "avg_review_rating", "cast_count",
            "genre_count", "studio_count", "keyword_count", "load_timestamp",
        )
    )
    return upsert(f"{SCHEMA}.gold_fact_movie_snapshot", rows, ["snapshot_key"])


def build_fact_review():
    create_table("gold_fact_review", """
        review_key BIGINT, movie_key BIGINT, movie_id BIGINT, reviewer_hash_id STRING,
        review_date_key INT, rating DOUBLE, weeks_since_release INT, batch_id STRING,
        source_extracted_at TIMESTAMP, load_timestamp TIMESTAMP
    """, "review_date_key, movie_key")

    rows = (
        current_movies.join(spark.table(f"{SCHEMA}.silver_movie_reviews"), "movie_id")
        .filter(F.col("review_id").isNotNull())
        .select(
            F.xxhash64("movie_id", "review_id").alias("review_key"),
            "movie_key", "movie_id", "reviewer_hash_id",
            F.coalesce(
                F.date_format(F.coalesce("created_at", "updated_at"), "yyyyMMdd").cast("int"), F.lit(-1)
            ).alias("review_date_key"),
            "rating", "weeks_since_release",
            F.col("source_batch_id").alias("batch_id"),
            "source_extracted_at",
            F.current_timestamp().alias("load_timestamp"),
        )
    )
    return upsert(f"{SCHEMA}.gold_fact_review", rows, ["review_key"])


step("gold_fact_movie_snapshot", build_fact_movie_snapshot)
step("gold_fact_review", build_fact_review)

In [ ]:
GOLD_TABLES = [
    "gold_dim_date", "gold_dim_movie", "gold_dim_genre", "gold_dim_studio",
    "gold_dim_person", "gold_dim_keyword", "gold_bridge_movie_genre",
    "gold_bridge_movie_studio", "gold_bridge_movie_cast", "gold_bridge_movie_keyword",
    "gold_fact_movie_snapshot", "gold_fact_review",
]

if run_optimize:
    for table in GOLD_TABLES:
        try:
            spark.sql(f"OPTIMIZE {SCHEMA}.{table}")
        except Exception as exc:
            print("OPTIMIZE skipped for", table, "->", str(exc)[:100])

display(spark.createDataFrame(
    [(t, spark.table(f"{SCHEMA}.{t}").count()) for t in GOLD_TABLES], "table_name STRING, row_count LONG"
))

In [ ]:
display(spark.sql(f"""
    WITH ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY movie_id ORDER BY snapshot_date_key DESC) AS rn
        FROM {SCHEMA}.gold_fact_movie_snapshot
    ), latest AS (SELECT * FROM ranked WHERE rn = 1)
    SELECT g.genre_name,
           COUNT(DISTINCT f.movie_key) AS movies,
           ROUND(SUM(f.revenue) / 1000000.0, 1) AS revenue_musd,
           ROUND(AVG(f.vote_average), 2) AS avg_vote
    FROM latest f
    JOIN {SCHEMA}.gold_bridge_movie_genre b ON b.movie_key = f.movie_key
    JOIN {SCHEMA}.gold_dim_genre g ON g.genre_key = b.genre_key
    GROUP BY g.genre_name
    ORDER BY revenue_musd DESC NULLS LAST
    LIMIT 10
"""))

display(spark.sql(f"""
    SELECT input_parameter, status, rows_processed, rows_inserted, rows_updated,
           ROUND(unix_timestamp(end_time) - unix_timestamp(start_time), 1) AS seconds
    FROM {LOGS}
    WHERE layer = 'Silver-to-Gold' AND batch_id = '{batch_id}'
    ORDER BY start_time
"""))